In [ ]:
RETRAIN = False


# Autoencoder x224 Main

This notebook is the curated submission-facing entry point for the convolutional autoencoder at 224x224 resolution on the shared anomaly benchmark.
It defaults to reusing the saved checkpoint and evaluation artifacts instead of retraining.


### Imports

This cell loads the libraries, repo-local modules, and path helpers used by the notebook.


In [1]:
from pathlib import Path
import json
import random
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
import torch
from IPython.display import display
from torch.utils.data import DataLoader
from tqdm import tqdm
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.models.autoencoder import ConvAutoencoder, build_autoencoder_from_config
from wafer_defect.scoring import absolute_error_map, squared_error_map, spatial_mean, topk_spatial_mean
from wafer_defect.training.autoencoder import run_autoencoder_epoch


### Run Controls

This cell defines the experiment config path and the main rerun flags. Leave `RETRAIN = False` to reuse saved artifacts when they already exist.


In [2]:
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/autoencoder/x224/main/train_config.toml'
EPOCHS_OVERRIDE = None
RERUN_SCORE_ABLATION = False
ANOMALY_SCORE_NAME = 'topk_abs_mean'
TOPK_RATIO = 0.01
config = load_toml(CONFIG_PATH)
if EPOCHS_OVERRIDE is not None:
    config['training']['epochs'] = int(EPOCHS_OVERRIDE)
config


{'run': {'output_dir': 'experiments/anomaly_detection/autoencoder/x224/main/artifacts/autoencoder_x224',
  'seed': 42},
 'data': {'metadata_csv': 'data/processed/x224/wm811k/metadata_50k_5pct.csv',
  'image_size': 224,
  'batch_size': 512,
  'num_workers': 8},
 'training': {'epochs': 20,
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'device': 'auto',
  'early_stopping_patience': 5,
  'early_stopping_min_delta': 5e-05,
  'checkpoint_every': 5,
  'resume_from': ''},
 'model': {'type': 'autoencoder',
  'latent_dim': 128,
  'use_batchnorm': False,
  'dropout_prob': 0.0}}

### Reproducibility And Helpers

This cell sets the random seed, resolves the execution device, and defines a helper for saving figures.


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def resolve_device(device_name: str) -> torch.device:
    if device_name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(device_name)

def save_figure(fig: plt.Figure, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print(f'Saved figure to {path}')
    return path

def warn_skip(message: str) -> None:
    print(f'[WARNING] {message}')

set_seed(int(config['run']['seed']))
device = resolve_device(config['training']['device'])
device


device(type='cuda')

### Metadata Check

This cell loads the configured metadata CSV so we can verify the split before building loaders.


In [4]:
metadata_path = REPO_ROOT / config['data']['metadata_csv']
image_size = int(config['data'].get('image_size', 64))
metadata_available = metadata_path.exists()
metadata = pd.DataFrame()
if metadata_available:
    metadata = pd.read_csv(metadata_path)
    display(metadata.head())
    display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
    display(metadata['is_anomaly'].value_counts().rename_axis('is_anomaly').to_frame('count'))
else:
    warn_skip(f'Metadata CSV not found: {metadata_path}. Dataset-backed cells will be skipped unless cached artifacts already exist.')


[WARNING] Metadata CSV not found: C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\data\processed\x224\wm811k\metadata_50k_5pct.csv. Dataset-backed cells will be skipped unless cached artifacts already exist.


### Data Loaders

This cell builds the train, validation, and test loaders used throughout the notebook.


In [5]:
train_dataset = None
val_dataset = None
test_dataset = None
train_loader = None
val_loader = None
test_loader = None
if metadata_available:
    train_dataset = WaferMapDataset(metadata_path, split='train', image_size=image_size)
    val_dataset = WaferMapDataset(metadata_path, split='val', image_size=image_size)
    test_dataset = WaferMapDataset(metadata_path, split='test', image_size=image_size)
    train_loader = DataLoader(train_dataset, batch_size=int(config['data']['batch_size']), shuffle=True, num_workers=int(config['data']['num_workers']))
    val_loader = DataLoader(val_dataset, batch_size=int(config['data']['batch_size']), shuffle=False, num_workers=int(config['data']['num_workers']))
    test_loader = DataLoader(test_dataset, batch_size=int(config['data']['batch_size']), shuffle=False, num_workers=int(config['data']['num_workers']))
    print(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')
else:
    warn_skip('WaferMapDataset/DataLoader construction is skipped because the metadata CSV is unavailable.')


[WARNING] WaferMapDataset/DataLoader construction is skipped because the metadata CSV is unavailable.


### Model Setup

This cell constructs the model and optimizer that will be used either for training or for loading an existing checkpoint.


In [6]:
model = ConvAutoencoder(latent_dim=int(config['model']['latent_dim']), image_size=image_size, use_batchnorm=bool(config['model'].get('use_batchnorm', False)), dropout_prob=float(config['model'].get('dropout_prob', 0.0))).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=float(config['training']['learning_rate']), weight_decay=float(config['training']['weight_decay']))
model


ConvAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=50176, out_features=128, bias=True)
    (8): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=50176, bias=True)
    (1): ReLU()
    (2): Unflatten(dim=1, unflattened_size=(64, 28, 28))
    (3): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): ReLU()
    (5): ConvTranspose2d(32, 16, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (6): ReLU()
    (7): ConvTranspose2d(16, 1, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (8): Sigmoid()
  )
)

### Training Or Artifact Reuse

This cell either trains the model or reuses the existing checkpoint and history files when they are already present.


In [7]:
history = []
output_dir = REPO_ROOT / config['run']['output_dir']
output_dir.mkdir(parents=True, exist_ok=True)
history_path = output_dir / 'results' / 'history.json'
best_model_path = output_dir / 'checkpoints' / 'best_model.pt'
resume_from = ''
best_epoch = 0
best_val_loss = float('nan')
stale_epochs = 0
best_state_dict = None
training_ran = False
evaluation_ready = False
artifacts_ready = best_model_path.exists() and history_path.exists()
if not RETRAIN and artifacts_ready:
    history = json.loads(history_path.read_text(encoding='utf-8'))
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    best_epoch = int(best_checkpoint.get('best_epoch', best_checkpoint.get('epoch', 0)))
    best_val_loss = float(best_checkpoint.get('best_val_loss', float('nan')))
    stale_epochs = int(best_checkpoint.get('stale_epochs', 0))
    best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f'Found existing artifacts in {output_dir}. Skipping training.')
elif RETRAIN and train_loader is not None and val_loader is not None:
    warn_skip('RETRAIN=True remains available, but this runtime-fix pass only supports artifact-backed notebook execution.')
elif RETRAIN:
    warn_skip('RETRAIN=True but the train/val datasets are unavailable in this notebook run.')
else:
    warn_skip('Saved training artifacts are missing and RETRAIN is False. Skipping training-backed cells.')


[WARNING] Saved training artifacts are missing and RETRAIN is False. Skipping training-backed cells.


### Training Curve

This cell displays the saved training history and exports the training-curve figure to the artifact folder.


In [8]:
if not history and history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
history_df = pd.DataFrame(history)
if history_df.empty:
    warn_skip('Training curves are unavailable because history.json is missing or empty.')
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train')
    ax.plot(history_df['epoch'], history_df['val_loss'], marker='o', label='val')
    ax.set_title('Autoencoder Training Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    save_figure(fig, output_dir / 'plots' / 'training_curves.png')
    plt.show()
    display(history_df.tail())


[WARNING] Training curves are unavailable because history.json is missing or empty.


### Persist Training Outputs

This cell writes training outputs only when a fresh training run was executed. If artifacts were reused, it reports that nothing was overwritten.


In [9]:
if training_ran:
    torch.save({'epoch': len(history), 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'config': config, 'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'stale_epochs': stale_epochs, 'history': history}, output_dir / 'checkpoints' / 'last_model.pt')
    if best_state_dict is not None:
        torch.save({'epoch': best_epoch, 'model_state_dict': best_state_dict, 'optimizer_state_dict': optimizer.state_dict(), 'config': config, 'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'stale_epochs': stale_epochs, 'history': history}, output_dir / 'checkpoints' / 'best_model.pt')
    with history_path.open('w', encoding='utf-8') as handle:
        json.dump(history, handle, indent=2)
    summary = {'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'epochs_ran': len(history), 'resumed_from': resume_from, 'training_ran': True}
    with (output_dir / 'results' / 'summary.json').open('w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=2)
    print(f'Saved outputs to {output_dir}')
else:
    print('Reused existing training artifacts; no training files were rewritten.')
    summary_path = output_dir / 'results' / 'summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {'best_epoch': best_epoch, 'best_val_loss': best_val_loss, 'epochs_ran': len(history), 'resumed_from': resume_from, 'training_ran': False}
summary


Reused existing training artifacts; no training files were rewritten.


{'best_epoch': 0,
 'best_val_loss': nan,
 'epochs_ran': 0,
 'resumed_from': '',
 'training_ran': False}

### Load Best Checkpoint And Score Test Split

This cell loads the best checkpoint and computes anomaly scores on the test split.


In [10]:
best_model_path = output_dir / 'checkpoints' / 'best_model.pt'
score_df = pd.DataFrame()
if not best_model_path.exists():
    warn_skip(f'Best autoencoder checkpoint not found: {best_model_path}. Skipping evaluation-backed cells.')
    evaluation_ready = False
elif test_loader is None:
    warn_skip('The test dataset is unavailable, so checkpoint-backed scoring is skipped for this cell.')
    evaluation_ready = False
else:
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    model.eval()

    def reconstruction_error(inputs: torch.Tensor, outputs: torch.Tensor, score_name: str=ANOMALY_SCORE_NAME) -> torch.Tensor:
        if score_name == 'mse_mean':
            return spatial_mean(squared_error_map(inputs, outputs))
        if score_name == 'topk_abs_mean':
            return topk_spatial_mean(absolute_error_map(inputs, outputs), topk_ratio=TOPK_RATIO)
        raise ValueError(f'Unsupported score_name: {score_name}')

    test_scores = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            scores = reconstruction_error(inputs, outputs, score_name=ANOMALY_SCORE_NAME).cpu().numpy()
            labels = labels.cpu().numpy()
            for score, label in zip(scores, labels):
                test_scores.append({'score': float(score), 'is_anomaly': int(label)})
    score_df = pd.DataFrame(test_scores)
    evaluation_ready = not score_df.empty
    display(score_df.head())


[WARNING] Best autoencoder checkpoint not found: C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\experiments\anomaly_detection\autoencoder\x224\main\artifacts\autoencoder_x224\checkpoints\best_model.pt. Skipping evaluation-backed cells.


### Validation Threshold

This cell computes the deployment threshold from validation-normal scores.


In [11]:
if not evaluation_ready:
    warn_skip('Validation-threshold selection is unavailable because evaluation scores were not generated.')
else:
    if not evaluation_ready:
        warn_skip('Validation-threshold selection is unavailable because evaluation scores were not generated.')
    else:
        val_scores = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                scores = reconstruction_error(inputs, outputs, score_name=ANOMALY_SCORE_NAME).cpu().numpy()
                val_scores.extend(scores.tolist())
        val_score_series = pd.Series(val_scores, name='val_score')
        threshold = float(val_score_series.quantile(0.95))
        print(f'Chosen threshold from validation normals (95th percentile, {ANOMALY_SCORE_NAME}): {threshold:.6f}')
        val_score_series.describe()


[WARNING] Validation-threshold selection is unavailable because evaluation scores were not generated.


### Metrics

This cell applies the validation-derived threshold, computes evaluation metrics, and saves the score table and metric summary.


In [12]:
if not evaluation_ready:
    warn_skip('Test metrics are unavailable because evaluation scores or thresholds are missing.')
else:
    if not evaluation_ready:
        warn_skip('Test metrics are unavailable because evaluation scores or thresholds are missing.')
    else:
        score_df['predicted_anomaly'] = (score_df['score'] > threshold).astype(int)
        precision = precision_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
        recall = recall_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
        f1 = f1_score(score_df['is_anomaly'], score_df['predicted_anomaly'], zero_division=0)
        auroc = roc_auc_score(score_df['is_anomaly'], score_df['score'])
        auprc = average_precision_score(score_df['is_anomaly'], score_df['score'])
        cm = confusion_matrix(score_df['is_anomaly'], score_df['predicted_anomaly'])
        metrics_df = pd.DataFrame([{'metric': 'score_name', 'value': ANOMALY_SCORE_NAME}, {'metric': 'precision', 'value': precision}, {'metric': 'recall', 'value': recall}, {'metric': 'f1', 'value': f1}, {'metric': 'auroc', 'value': auroc}, {'metric': 'auprc', 'value': auprc}, {'metric': 'threshold', 'value': threshold}])
        display(metrics_df)
        cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
        display(cm_df)
        fig, ax = plt.subplots(figsize=(5, 4))
        heatmap = ax.imshow(cm_df.to_numpy(), cmap='Blues')
        ax.set_xticks(range(cm_df.shape[1]), cm_df.columns)
        ax.set_yticks(range(cm_df.shape[0]), cm_df.index)
        ax.set_title('Confusion Matrix At Validation Threshold')
        ax.set_xlabel('Predicted label')
        ax.set_ylabel('True label')
        for row_idx in range(cm_df.shape[0]):
            for col_idx in range(cm_df.shape[1]):
                value = int(cm_df.iat[row_idx, col_idx])
                ax.text(col_idx, row_idx, str(value), ha='center', va='center', color='black')
        fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        save_figure(fig, output_dir / 'plots' / 'confusion_matrix.png')
        plt.show()
        score_df.to_csv(output_dir / 'results' / 'test_scores.csv', index=False)
        metrics_df.to_csv(output_dir / 'results' / 'metrics.csv', index=False)


[WARNING] Test metrics are unavailable because evaluation scores or thresholds are missing.


### Threshold Sweep Plot

This cell compares precision, recall, and F1 across score thresholds, then saves both the table and the figure.


In [13]:
if not evaluation_ready:
    warn_skip('Threshold sweep is unavailable because evaluation scores or thresholds are missing.')
else:
    if not evaluation_ready:
        warn_skip('Threshold sweep is unavailable because evaluation scores or thresholds are missing.')
    else:
        precision_curve, recall_curve, pr_thresholds = precision_recall_curve(score_df['is_anomaly'], score_df['score'])
        threshold_sweep_df = pd.DataFrame({'threshold': pr_thresholds, 'precision': precision_curve[:-1], 'recall': recall_curve[:-1]})
        threshold_sweep_df['f1'] = 2 * threshold_sweep_df['precision'] * threshold_sweep_df['recall'] / (threshold_sweep_df['precision'] + threshold_sweep_df['recall'] + 1e-12)
        threshold_sweep_df['predicted_anomalies'] = [int((score_df['score'] > t).sum()) for t in threshold_sweep_df['threshold']]
        best_f1_row = threshold_sweep_df.loc[threshold_sweep_df['f1'].idxmax()]
        threshold_sweep_df.to_csv(output_dir / 'results' / 'threshold_sweep.csv', index=False)
        display(threshold_sweep_df.sort_values('f1', ascending=False).head(10))
        print(f"Best F1 threshold: {best_f1_row['threshold']:.6f} | precision={best_f1_row['precision']:.4f}, recall={best_f1_row['recall']:.4f}, f1={best_f1_row['f1']:.4f}")
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
        ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
        ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
        ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
        ax.axvline(best_f1_row['threshold'], color='green', linestyle=':', label=f"best f1 threshold = {best_f1_row['threshold']:.4f}")
        ax.set_xlabel('Anomaly-score threshold')
        ax.set_ylabel('Metric value')
        ax.set_title(f'Threshold Sweep on Test Split ({ANOMALY_SCORE_NAME})')
        ax.legend()
        save_figure(fig, output_dir / 'plots' / 'threshold_sweep.png')
        plt.show()


[WARNING] Threshold sweep is unavailable because evaluation scores or thresholds are missing.


### Score Distribution Plot

This cell visualizes the test-score distribution for normal and anomalous wafers and saves the histogram figure.


In [14]:
if not evaluation_ready:
    warn_skip('Score-distribution plotting is unavailable because evaluation scores or thresholds are missing.')
else:
    if not evaluation_ready:
        warn_skip('Score-distribution plotting is unavailable because evaluation scores or thresholds are missing.')
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(score_df[score_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal')
        ax.hist(score_df[score_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly')
        ax.axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
        ax.set_title(f'Anomaly Score on Test Split ({ANOMALY_SCORE_NAME})')
        ax.set_xlabel(f'Per-sample score: {ANOMALY_SCORE_NAME}')
        ax.set_ylabel('Count')
        ax.legend()
        plt.tight_layout()
        save_figure(fig, output_dir / 'plots' / 'score_distribution.png')
        plt.show()


[WARNING] Score-distribution plotting is unavailable because evaluation scores or thresholds are missing.


### Reconstruction Examples

This cell shows a small set of input and reconstruction pairs and saves the figure.


In [15]:
if not evaluation_ready:
    warn_skip('Reconstruction examples are unavailable because evaluation scores or datasets are missing.')
else:
    if not evaluation_ready:
        warn_skip('Reconstruction examples are unavailable because evaluation scores or datasets are missing.')
    else:
        normal_test_idx = score_df[score_df['is_anomaly'] == 0].index[:4].tolist()
        anomaly_test_idx = score_df[score_df['is_anomaly'] == 1].index[:4].tolist()
        selected_indices = normal_test_idx + anomaly_test_idx
        fig, axes = plt.subplots(len(selected_indices), 2, figsize=(6, 2.3 * len(selected_indices)))
        with torch.no_grad():
            for row_idx, sample_idx in enumerate(selected_indices):
                input_tensor, label = test_dataset[sample_idx]
                output_tensor = model(input_tensor.unsqueeze(0).to(device)).squeeze(0).cpu()
                title_prefix = 'anomaly' if int(label) == 1 else 'normal'
                score = score_df.iloc[sample_idx]['score']
                axes[row_idx, 0].imshow(input_tensor.squeeze(0), cmap='viridis')
                axes[row_idx, 0].set_title(f'Input: {title_prefix}')
                axes[row_idx, 0].axis('off')
                axes[row_idx, 1].imshow(output_tensor.squeeze(0), cmap='viridis')
                axes[row_idx, 1].set_title(f'Recon, score={score:.4f}')
                axes[row_idx, 1].axis('off')
        plt.tight_layout()
        save_figure(fig, output_dir / 'plots' / 'reconstruction_examples.png')
        plt.show()


[WARNING] Reconstruction examples are unavailable because evaluation scores or datasets are missing.


### Failure Tables

This cell builds the error-analysis table and saves the detailed failure-analysis CSVs for later reference.


In [16]:
if not evaluation_ready:
    warn_skip('Failure-analysis tables are unavailable because evaluation scores or thresholds are missing.')
else:
    if not evaluation_ready:
        warn_skip('Failure-analysis tables are unavailable because evaluation scores or thresholds are missing.')
    else:
        analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
        analysis_df = pd.concat([analysis_df, score_df[['score', 'predicted_anomaly']].reset_index(drop=True)], axis=1)
        analysis_df['error_type'] = 'tn'
        analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
        analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
        analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'
        analysis_df['correct'] = analysis_df['is_anomaly'] == analysis_df['predicted_anomaly']
        error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])
        defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
        defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']
        fp_defect_df = analysis_df[analysis_df['error_type'] == 'fp'].groupby('defect_type').agg(count=('defect_type', 'size'), mean_score=('score', 'mean')).sort_values(['count', 'mean_score'], ascending=[False, False])
        display(error_summary_df)
        display(defect_recall_df)
        display(fp_defect_df)
        analysis_df.head()
        analysis_df.to_csv(output_dir / 'results' / 'failure_analysis.csv', index=False)
        error_summary_df.to_csv(output_dir / 'results' / 'failure_error_summary.csv')
        defect_recall_df.to_csv(output_dir / 'results' / 'failure_defect_recall.csv')
        fp_defect_df.to_csv(output_dir / 'results' / 'failure_false_positive_breakdown.csv')


[WARNING] Failure-analysis tables are unavailable because evaluation scores or thresholds are missing.


### Failure Examples

This cell visualizes representative false positives, false negatives, true positives, and true negatives and saves each figure.


### Score Ablation

This cell computes anomaly scores under seven different scoring rules and selects the best performer.


In [17]:
if not evaluation_ready:
    warn_skip('Score ablation is unavailable because the saved evaluation artifacts are missing.')
else:
    if not evaluation_ready:
        warn_skip('Score ablation is unavailable because the saved evaluation artifacts are missing.')
    else:
        score_ablation_path = output_dir / 'results' / 'score_ablation.csv'
        if RERUN_SCORE_ABLATION:
            print('Computing score ablation across 7 scoring rules...')

            def score_mse_mean(inputs, outputs):
                return spatial_mean(squared_error_map(inputs, outputs))

            def score_max_abs(inputs, outputs):
                ae = absolute_error_map(inputs, outputs)
                return ae.view(ae.shape[0], -1).max(dim=1)[0]

            def score_topk_abs_mean(inputs, outputs, topk_ratio=0.01):
                return topk_spatial_mean(absolute_error_map(inputs, outputs), topk_ratio=topk_ratio)

            def score_mae_mean(inputs, outputs):
                ae = absolute_error_map(inputs, outputs)
                return spatial_mean(ae)

            def score_foreground_mse(inputs, outputs):
                se = squared_error_map(inputs, outputs)
                mask = (inputs > 0.1).float()
                masked_se = se * mask
                return (masked_se.view(masked_se.shape[0], -1).sum(dim=1) + 1e-08) / (mask.view(mask.shape[0], -1).sum(dim=1) + 1e-08)

            def score_foreground_mae(inputs, outputs):
                ae = absolute_error_map(inputs, outputs)
                mask = (inputs > 0.1).float()
                masked_ae = ae * mask
                return (masked_ae.view(masked_ae.shape[0], -1).sum(dim=1) + 1e-08) / (mask.view(mask.shape[0], -1).sum(dim=1) + 1e-08)

            def score_pooled_mae_mean(inputs, outputs):
                ae = absolute_error_map(inputs, outputs)
                pooled = torch.nn.functional.max_pool2d(ae, kernel_size=2, stride=2)
                return spatial_mean(pooled)
            score_functions = {'mse_mean': lambda i, o: score_mse_mean(i, o), 'max_abs': lambda i, o: score_max_abs(i, o), 'topk_abs_mean': lambda i, o: score_topk_abs_mean(i, o, topk_ratio=0.01), 'mae_mean': lambda i, o: score_mae_mean(i, o), 'foreground_mse': lambda i, o: score_foreground_mse(i, o), 'foreground_mae': lambda i, o: score_foreground_mae(i, o), 'pooled_mae_mean': lambda i, o: score_pooled_mae_mean(i, o)}
            print('Computing validation thresholds...')
            val_thresholds = {}
            with torch.no_grad():
                for score_name, score_fn in score_functions.items():
                    val_scores = []
                    for inputs, _ in val_loader:
                        inputs = inputs.to(device)
                        outputs = model(inputs)
                        scores = score_fn(inputs, outputs).cpu().numpy()
                        val_scores.extend(scores.tolist())
                    val_thresholds[score_name] = float(np.percentile(val_scores, 95))
            print('Computing test scores and metrics...')
            results = []
            with torch.no_grad():
                for score_name, score_fn in score_functions.items():
                    test_scores = []
                    labels_list = []
                    for inputs, labels in test_loader:
                        inputs = inputs.to(device)
                        outputs = model(inputs)
                        scores = score_fn(inputs, outputs).cpu().numpy()
                        test_scores.extend(scores.tolist())
                        labels_list.extend(labels.cpu().numpy().tolist())
                    test_scores = np.array(test_scores)
                    labels_array = np.array(labels_list)
                    threshold = val_thresholds[score_name]
                    predictions = (test_scores > threshold).astype(int)
                    precision = precision_score(labels_array, predictions, zero_division=0)
                    recall = recall_score(labels_array, predictions, zero_division=0)
                    f1 = f1_score(labels_array, predictions, zero_division=0)
                    auroc = roc_auc_score(labels_array, test_scores)
                    auprc = average_precision_score(labels_array, test_scores)
                    best_f1 = 0
                    for t in np.percentile(test_scores, np.linspace(0, 100, 100)):
                        preds = (test_scores > t).astype(int)
                        best_f1 = max(best_f1, f1_score(labels_array, preds, zero_division=0))
                    results.append({'score': score_name, 'val_threshold_f1': f1, 'precision': precision, 'recall': recall, 'f1': f1, 'auroc': auroc, 'auprc': auprc, 'best_sweep_f1': best_f1})
            ablation_df = pd.DataFrame(results).sort_values('f1', ascending=False)
            ablation_df.to_csv(score_ablation_path, index=False)
            print(f'Saved score ablation to {score_ablation_path}')
        elif score_ablation_path.exists():
            ablation_df = pd.read_csv(score_ablation_path)
            print(f'Loaded cached score ablation from {score_ablation_path}')
        else:
            print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
        display(ablation_df)
        ablation_df_sorted = ablation_df.sort_values('f1', ascending=True)
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes[0, 0].barh(ablation_df_sorted['score'], ablation_df_sorted['f1'], color='steelblue')
        axes[0, 0].set_xlabel('F1 (at validation threshold)')
        axes[0, 0].set_title('Validation-Threshold F1 by Score')
        axes[0, 0].grid(axis='x', alpha=0.3)
        axes[0, 1].barh(ablation_df_sorted['score'], ablation_df_sorted['auroc'], color='orange')
        axes[0, 1].set_xlabel('AUROC')
        axes[0, 1].set_title('AUROC by Score')
        axes[0, 1].grid(axis='x', alpha=0.3)
        axes[1, 0].barh(ablation_df_sorted['score'], ablation_df_sorted['auprc'], color='green')
        axes[1, 0].set_xlabel('AUPRC')
        axes[1, 0].set_title('AUPRC by Score')
        axes[1, 0].grid(axis='x', alpha=0.3)
        axes[1, 1].barh(ablation_df_sorted['score'], ablation_df_sorted['best_sweep_f1'], color='red')
        axes[1, 1].set_xlabel('Best Sweep F1')
        axes[1, 1].set_title('Best Sweep F1 by Score')
        axes[1, 1].grid(axis='x', alpha=0.3)
        plt.tight_layout()
        save_figure(fig, output_dir / 'plots' / 'score_ablation_summary.png')
        plt.show()
        best_row = ablation_df.iloc[0]
        print(f"\nSelected score: {best_row['score']}")
        print(f"  Val-threshold F1: {best_row['f1']:.6f}")
        print(f"  Precision: {best_row['precision']:.6f}")
        print(f"  Recall: {best_row['recall']:.6f}")
        print(f"  AUROC: {best_row['auroc']:.6f}")
        print(f"  AUPRC: {best_row['auprc']:.6f}")
        print(f"  Best sweep F1: {best_row['best_sweep_f1']:.6f}")


[WARNING] Score ablation is unavailable because the saved evaluation artifacts are missing.


In [18]:
if not evaluation_ready:
    warn_skip('Failure-example rendering is unavailable because the analysis dataframe is empty.')
else:
    if not evaluation_ready:
        warn_skip('Failure-example rendering is unavailable because the analysis dataframe is empty.')
    else:
        def show_error_examples(error_type: str, n_examples: int=6, score_order: str='desc') -> pd.DataFrame:
            subset = analysis_df[analysis_df['error_type'] == error_type].copy()
            if subset.empty:
                print(f'No samples found for error_type={error_type!r}.')
                return subset
            ascending = score_order == 'asc'
            subset = subset.sort_values('score', ascending=ascending).head(n_examples)
            fig, axes = plt.subplots(len(subset), 3, figsize=(9, 2.8 * len(subset)))
            if len(subset) == 1:
                axes = np.expand_dims(axes, axis=0)
            with torch.no_grad():
                for row_idx, (sample_idx, row) in enumerate(subset.iterrows()):
                    input_tensor, label = test_dataset[sample_idx]
                    output_tensor = model(input_tensor.unsqueeze(0).to(device)).squeeze(0).cpu()
                    error_map = absolute_error_map(input_tensor.unsqueeze(0), output_tensor.unsqueeze(0)).squeeze(0).squeeze(0).cpu()
                    axes[row_idx, 0].imshow(input_tensor.squeeze(0), cmap='viridis')
                    axes[row_idx, 0].set_title(f"Input\nlabel={int(label)} | defect={row.get('defect_type', 'unknown')}")
                    axes[row_idx, 0].axis('off')
                    axes[row_idx, 1].imshow(output_tensor.squeeze(0), cmap='viridis')
                    axes[row_idx, 1].set_title(f"Recon\nscore={row['score']:.4f} | pred={row['predicted_anomaly']}")
                    axes[row_idx, 1].axis('off')
                    axes[row_idx, 2].imshow(error_map, cmap='magma')
                    axes[row_idx, 2].set_title(f'Abs error map\n{error_type.upper()} sample #{sample_idx}')
                    axes[row_idx, 2].axis('off')
            plt.tight_layout()
            save_figure(fig, output_dir / 'plots' / f'failure_examples_{error_type}.png')
            plt.show()
            return subset[['defect_type', 'score', 'predicted_anomaly', 'error_type']]
        display(show_error_examples('fp', n_examples=6, score_order='desc'))
        display(show_error_examples('fn', n_examples=6, score_order='asc'))
        display(show_error_examples('tp', n_examples=4, score_order='desc'))
        display(show_error_examples('tn', n_examples=4, score_order='asc'))


[WARNING] Failure-example rendering is unavailable because the analysis dataframe is empty.


### Expanded Holdout Evaluation (70 k normals / 3.5 k defects)

Re-evaluates the best checkpoint on the secondary holdout split (70 000 normal + 3 500 defect test wafers).
The x224 holdout arrays were generated natively from the raw pickle at 224x224 using the same seed and sampling logic as the x64 holdout, so the same wafers are selected.
The validation-derived threshold (`threshold`) computed above is reused without modification.


In [19]:
if not evaluation_ready:
    warn_skip('Holdout evaluation is skipped because the main evaluation is unavailable or the holdout metadata CSV is missing.')
else:
    if not evaluation_ready:
        warn_skip('Holdout evaluation is skipped because the main evaluation is unavailable or the holdout metadata CSV is missing.')
    else:
        HOLDOUT_METADATA_PATH = REPO_ROOT / 'data/processed/x224/wm811k/metadata_50k_5pct_holdout70k_3p5k.csv'
        HOLDOUT_OUTPUT_DIR = output_dir / 'results' / 'holdout70k_3p5k'
        HOLDOUT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        holdout_test_dataset = WaferMapDataset(HOLDOUT_METADATA_PATH, split='test', image_size=image_size)
        holdout_loader = DataLoader(holdout_test_dataset, batch_size=int(config['data']['batch_size']), shuffle=False, num_workers=int(config['data']['num_workers']))
        holdout_meta_df = holdout_test_dataset.metadata.reset_index(drop=True)
        print(f"Holdout test: {(holdout_meta_df['is_anomaly'] == 0).sum()} normals, {(holdout_meta_df['is_anomaly'] == 1).sum()} defects")
        model.eval()
        holdout_rows = []
        with torch.no_grad():
            for inputs, labels in tqdm(holdout_loader, desc='Scoring holdout'):
                inputs = inputs.to(device)
                outputs = model(inputs)
                scores = reconstruction_error(inputs, outputs, score_name=ANOMALY_SCORE_NAME).cpu().numpy()
                for score, label in zip(scores, labels.cpu().numpy()):
                    holdout_rows.append({'score': float(score), 'is_anomaly': int(label)})
        holdout_score_df = pd.DataFrame(holdout_rows)
        holdout_score_df['predicted_anomaly'] = (holdout_score_df['score'] > threshold).astype(int)
        precision_h = precision_score(holdout_score_df['is_anomaly'], holdout_score_df['predicted_anomaly'], zero_division=0)
        recall_h = recall_score(holdout_score_df['is_anomaly'], holdout_score_df['predicted_anomaly'], zero_division=0)
        f1_h = f1_score(holdout_score_df['is_anomaly'], holdout_score_df['predicted_anomaly'], zero_division=0)
        auroc_h = roc_auc_score(holdout_score_df['is_anomaly'], holdout_score_df['score'])
        auprc_h = average_precision_score(holdout_score_df['is_anomaly'], holdout_score_df['score'])
        cm_h = confusion_matrix(holdout_score_df['is_anomaly'], holdout_score_df['predicted_anomaly'])
        print(f'Holdout F1:        {f1_h:.4f}')
        print(f'Holdout AUROC:     {auroc_h:.4f}')
        print(f'Holdout AUPRC:     {auprc_h:.4f}')
        print(f'Holdout Precision: {precision_h:.4f}  Recall: {recall_h:.4f}')
        prec_curve_h, rec_curve_h, pr_thresholds_h = precision_recall_curve(holdout_score_df['is_anomaly'], holdout_score_df['score'])
        sweep_df_h = pd.DataFrame({'threshold': pr_thresholds_h, 'precision': prec_curve_h[:-1], 'recall': rec_curve_h[:-1]})
        sweep_df_h['f1'] = 2 * sweep_df_h['precision'] * sweep_df_h['recall'] / (sweep_df_h['precision'] + sweep_df_h['recall'] + 1e-12)
        sweep_df_h['predicted_anomalies'] = [int((holdout_score_df['score'] > t).sum()) for t in sweep_df_h['threshold']]
        best_h = sweep_df_h.loc[sweep_df_h['f1'].idxmax()]
        print(f"Best sweep F1:     {best_h['f1']:.4f}  (threshold={best_h['threshold']:.6f})")
        holdout_score_df['defect_type'] = holdout_meta_df['defect_type'].values
        defect_breakdown_h = holdout_score_df[holdout_score_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
        defect_breakdown_h['recall'] = defect_breakdown_h['detected'] / defect_breakdown_h['count']
        display(defect_breakdown_h)
        holdout_score_df.to_csv(HOLDOUT_OUTPUT_DIR / 'test_scores.csv', index=False)
        sweep_df_h.to_csv(HOLDOUT_OUTPUT_DIR / 'threshold_sweep.csv', index=False)
        defect_breakdown_h.to_csv(HOLDOUT_OUTPUT_DIR / 'defect_breakdown.csv')
        cm_df_h = pd.DataFrame(cm_h, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
        cm_df_h.to_csv(HOLDOUT_OUTPUT_DIR / 'confusion_matrix.csv')
        summary_h = {'model_type': 'autoencoder_x224', 'score_name': ANOMALY_SCORE_NAME, 'checkpoint': str(best_model_path), 'metadata_csv': str(HOLDOUT_METADATA_PATH), 'threshold_quantile': 0.95, 'threshold': threshold, 'metrics_at_validation_threshold': {'threshold': threshold, 'precision': precision_h, 'recall': recall_h, 'f1': f1_h, 'auroc': auroc_h, 'auprc': auprc_h, 'predicted_anomalies': int(holdout_score_df['predicted_anomaly'].sum()), 'confusion_matrix': cm_h.tolist()}, 'best_threshold_sweep': {'threshold': float(best_h['threshold']), 'precision': float(best_h['precision']), 'recall': float(best_h['recall']), 'f1': float(best_h['f1']), 'predicted_anomalies': int(best_h['predicted_anomalies'])}, 'counts': {'val_normal': 5000, 'test_normal': int((holdout_meta_df['is_anomaly'] == 0).sum()), 'test_anomaly': int((holdout_meta_df['is_anomaly'] == 1).sum())}}
        with (HOLDOUT_OUTPUT_DIR / 'summary.json').open('w', encoding='utf-8') as fh:
            json.dump(summary_h, fh, indent=2)
        print(f'\nSaved holdout results to {HOLDOUT_OUTPUT_DIR}')


[WARNING] Holdout evaluation is skipped because the main evaluation is unavailable or the holdout metadata CSV is missing.
